# Phase 2: From Codes to Themes

An initial thematic map is generated by prompting the LLM on a random
batch of 12 codes together with their associated justifications to provide
analytical context.

The remaining codes are then processed sequentially together with their
associated justifications and interview segments for contextual grounding.
Using a domain-specific prompting strategy, the LLM assigns each code to
an existing theme or marks it as non-applicable.

Unassigned codes are continuously accumulated during processing. When the
pool reaches 20 codes, the thematic map is updated by prompting the LLM on
a new random subset of 12 codes. This allows for iterative refinement and
expansion of the thematic structure.

During thematic map updates, previously generated themes are provided to
the model as contextual guidance. The model is instructed to introduce new
themes only when the current data contains patterns not already represented
in the existing thematic structure

The outputs of this phase consist of:
- Themes
- Theme descriptions
- Justifications explaining why each code is assigned to a given theme

Codes that remain unassigned are retained in a separate pool for
documentation purposes, indicating that they may have limited empirical
support in the data.



In [ ]:
import outlines
import transformers
import torch

import os
import torch
from pydantic import BaseModel, Field
from typing import List

from datetime import datetime
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline


### Read Data

In [ ]:
import pandas as pd
import json

df_merge = pd.read_parquet(data_path / "currentTEST.parquet")

df_merge.columns



Index(['unit_id', 'int_id', 'interview_excerpt', 'segment_id', 'codes',
       'justifications', 'status', 'error', 'code_id', 'code_uid',
       'code_label', 'code_label_justification', 'justification_label',
       'justification_eval_explanation', 'refine_action_x', 'refine_error',
       'codes_final', 'justifications_final', 'refine_action_y'],
      dtype='object')

### Subset data for merge of codes (only code_uid, codes_final, justifications_final, interview_excerpt)

In [ ]:
#Filter codes and justifications that are not relevant (bc they cannot become themes)
df_filtered = df_merge[
    (df_merge["codes_final"] != "not relevant") &
    (df_merge["justifications_final"] != "not relevant") &
    (df_merge["codes_final"].notna()) &
    (df_merge["justifications_final"].notna())
]

In [ ]:
data = df_filtered[[
    "code_uid", # anchor
    "codes_final", #main analytical object
    "justifications_final", #for context and grounding
    "interview_excerpt",# for context and grounding
    "int_id" #for merging
]]

### Pipeline 1: Function to subset 12 random sample codes --> generate initial thematic map






**STEP 1:** Function that samples and batches 12 data points to feed to LLM

In [ ]:
def build_batch_from_uids(uid_pool, data, batch_size=12):
    import random

    batch_size = min(batch_size, len(uid_pool))

    # 🔹 sample uids
    selected_uids = random.sample(uid_pool, batch_size)

    # 🔹 remove from pool (no replacement)
    for uid in selected_uids:
        uid_pool.remove(uid)

    # 🔹 filter rows using pandas
    df_batch = data[data["code_uid"].isin(selected_uids)]

    # 🔹 optional: preserve original sampling order
    df_batch = df_batch.set_index("code_uid").loc[selected_uids].reset_index()

    # 🔹 format
    formatted_items = []
    for idx, (_, row) in enumerate(df_batch.iterrows(), start=1):
        formatted_items.append(
            f"[{idx}] Code: {row['codes_final']}\n"
            f"Justification: {row['justifications_final']}\n"
            f"Excerpt: {row['interview_excerpt']}"
        )

    return {
        "batch_text": "\n\n".join(formatted_items),
        "uids": selected_uids
    }

### Load LLM model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_id) #Mistral's tokenizer

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto" #A100 or H100
)

print("Mistral-7B-Instruct-v0.3 is loaded.")

### LLM configurations w. Outlines

In [ ]:
from transformers import GenerationConfig

model.generation_config = GenerationConfig.from_model_config(model.config)

model.generation_config.max_new_tokens = 800 #max token output
#model.generation_config.min_new_tokens = 40 #min new tokens
model.generation_config.temperature = 0.0 # Temp set to 0.0 because it is overridden by Outlines in structured output
model.generation_config.do_sample = False #LLM chooses most likely next token = False
model.generation_config.pad_token_id = tokenizer.eos_token_id # Mistral does not have a pad token, so we pad with EOS
model.generation_config.eos_token_id = tokenizer.eos_token_id #Mistral's end-of-sequence token

### Import Outlines for validation of output

In [ ]:
import outlines

outlines_model = outlines.from_transformers(model, tokenizer)

**STEP 2**: Analysis prompt for initial thematic map

### Format revise stage prompt in INST (chat_template)

In [ ]:
def format_prompt(content, tokenizer): #Taking analysis prompt and sending it to model in instruct mode
    messages = [
        {"role": "user", "content": content}
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

### Analysis prompt for iterative theme construction - second part of codes to themes

In [ ]:
def get_initial_theme_map_prompt(batch, existing_candidate_themes=None): #existing themes is a df
  existing_themes_block = ""

  if existing_candidate_themes is not None and not existing_candidate_themes.empty:
      # Format themes list for the model
      formatted_themes = "\n".join([f"- {row['theme_label']}: {row['theme_description']}" for _, row in existing_candidate_themes.iterrows()])

      existing_themes_block = f"""
### EXISTING THEMES

The following themes have already been identified in earlier batches:

{formatted_themes}

### INTEGRATION INSTRUCTIONS

- Compare the patterns in the current data with the existing themes
- If a pattern is already represented by an existing theme, do not create a new theme for it
- Only introduce a theme if it captures a pattern that is not already covered
- Do NOT repeat, restate, or slightly rephrase existing themes
- If all patterns in the data are already covered, return an empty list
"""

  content = f"""
### TASK
You are an expert qualitative researcher conducting Thematic Analysis (Braun & Clarke 2006).

Your task is to construct a provisional thematic map based on a batch of coded data extracts.
Identify patterns of shared underlying meaning across the data.

IMPORTANT:
- The goal is to generate plausible, provisional themes (organizing concepts)
- This is an exploratory step — themes do NOT need to be complete or final
- It is better to return fewer, stronger themes than many weak ones
- Do NOT collapse multiple distinct patterns into a single broad theme
- Return between 3 and 6 themes when no existing themes are provided
- When existing themes are provided, return only new themes (possibly zero)


### RESEARCH OBJECTIVE
The overall research objective is to understand the interviewees’ individual experiences of transformations in Danish society
and family life from the 1960s–1980s, including changing gender roles, labour participation, and family structures.


{existing_themes_block}


### DATA (CODED EXTRACTS)

Each item includes:
- a code (analytical label)
- a justification (interpretation)
- an interview excerpt (empirical grounding)

{batch}


### FORM REQUIREMENTS

A theme captures a meaningful and important pattern in the data in relation to the research objective.

- Theme_label: captures the essence of the pattern and clearly distinguishes it from other themes (2–6 words).
- Theme_description: describes the content of the theme in alignment with the nuances of the data (2–3 sentences).


### INSTRUCTIONS

1. Identify patterns of shared meaning across the coded extracts
2. Construct candidate themes that capture these patterns
3. Each theme should:
   - reflect a meaningful pattern across multiple data points
   - capture an underlying organizing concept (not just surface similarity)

4. Use the codes, justifications, and excerpts as analytical input for identifying patterns

5. It is acceptable that:
   - some codes are not reflected in any theme
   - themes may overlap slightly, but should remain analytically distinguishable
   - themes are incomplete or tentative


### THEME REQUIREMENTS

- Theme labels must be concise and analytically meaningful.
- Theme descriptions must explain the underlying pattern in the data.
- Themes must operate at a higher level of abstraction than individual codes.
- Themes should capture an underlying pattern or organizing concept across multiple data points.
- Avoid themes that simply group codes or resemble broad topics (e.g., "education", "work", "family").
- Each theme should reflect how experiences are structured or understood, not just what the topic is.


### OUTPUT FORMAT

Return ONLY valid JSON with this structure:

{{
  "themes": [
    {{
      "theme_label": "string",
      "theme_description": "string"
    }}
  ]
}}


### RULES

- Do NOT include any text outside the JSON
- Do NOT include explanations
- Do NOT include trailing commas
- Do NOT refer to codes by name or label
"""

  return content

### **STEP 3:** Defining  output schema with Pydantic  



In [ ]:
# Schema for post-validation (after LLM output)
import re
from pydantic import BaseModel, Field, field_validator
from typing import List


class CandidateTheme(BaseModel):
    theme_label: str = Field(..., min_length=3, max_length=100)
    theme_description: str = Field(..., min_length=10, max_length=300)

    @field_validator("theme_label", "theme_description")
    def must_contain_letters(cls, v):
        if not re.search(r"[a-zA-Z]", v):
            raise ValueError("Must contain at least one letter")
        return v


class InitialThemeMap(BaseModel):
    themes: List[CandidateTheme]

In [ ]:
#Function for retries
import time


def run_with_retry_initialThemes(outlines_model, prompt_inst, response_model, max_retries=3, retry_delay=1):

    for attempt in range(max_retries):
        try:
            result = outlines_model(prompt_inst, response_model)

            return result  # outlines returnerer allerede parsed object

        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")

            if attempt < max_retries - 1:
                time.sleep(retry_delay)
                continue

            raise ValueError(f"Failed after {max_retries} attempts: {e}")

In [ ]:
#Function for normalizing theme labels
import re

def normalize_theme_format(label: str) -> str:
    if not label:
        return None

    label = label.strip()

    # collapse whitespace
    label = re.sub(r"\s+", " ", label)

    # remove trailing
    label = label.strip(" .,-")

    # Capitalize first letter
    if label:
        label = label[0].upper() + label[1:]

    return label

In [ ]:
def run_theme_extraction(
    data,
    uid_pool,
    outlines_model,
    tokenizer,
    existing_candidate_themes=None,
    batch_size=12,
    return_df=True,
    verbose=True
):
    start_total = time.time()

    try:
        # 1. Build batch
        batch =  build_batch_from_uids(uid_pool,data, batch_size=batch_size)

        if verbose:
            print("\n--- BATCH SENT TO LLM ---")

        # 🔹 Build prompt
        content = get_initial_theme_map_prompt(batch["batch_text"], existing_candidate_themes)

        # 🔹 Format prompt
        prompt_inst = format_prompt(content, tokenizer)

        # 🔹 Run model
        validated_raw = run_with_retry_initialThemes(
            outlines_model,
            prompt_inst,
            InitialThemeMap
        )

        # 🔹 Parse output
        if isinstance(validated_raw, str):
            validated_dict = json.loads(validated_raw)
        else:
            validated_dict = validated_raw

        validated = InitialThemeMap(**validated_dict)

        if verbose:
            print("\n--- MODEL OUTPUT ---")
            print(validated)

        # 🔹 Process output
        results = []

        if not validated.themes:
            print("WARNING: No themes returned")

        for theme in validated.themes:
            theme_label = normalize_theme_format(theme.theme_label)

            results.append({
                "theme_label": theme_label,
                "theme_description": theme.theme_description.strip()
            })

    except Exception as e:
        import traceback
        print(f"Pipeline failed: {e}")
        traceback.print_exc()
        results = []

    total_time = time.time() - start_total

    if verbose:
        print(f"\nTotal time: {total_time:.2f}s")

    if return_df:
        return pd.DataFrame(results)
    else:
        return results

### Run pipeline - call functions


In [ ]:
uid_pool = data["code_uid"].tolist()

df_initial_themes = run_theme_extraction(
    data,
    uid_pool,
    outlines_model,
    tokenizer,
    batch_size=12,
    return_df=True,
    verbose=True
)
print(df_initial_themes)

# Second part of pipeline for theme construction

### Analysis prompt for iterative theme construction

In [ ]:
def get_theme_prompt(codes_final, code_uid, justifications_final, interview_excerpt, themes):

    content = f"""
### TASK
You are an expert qualitative researcher conducting Thematic Analysis (Braun & Clarke 2006).

Your task is to evaluate whether the code reflects a pattern of meaning that is already captured by any of the existing themes.

IMPORTANT:
- Not all codes will fit an existing theme
- It is acceptable and expected that some codes do NOT fit any theme
- Avoid forcing a match where conceptual alignment is weak


### RESEARCH OBJECTIVE
The overall research objective is to understand the interviewees’ individual experiences of transformations in Danish society
and family life from the 1960s-1980s.


### CODE
Code: {codes_final}
Code UID: {code_uid}
Justification: {justifications_final}
Interview excerpt: {interview_excerpt}


### EXISTING THEMES
{themes}


### ANALYTICAL INSTRUCTIONS

1. Identify the underlying pattern of meaning in the code
2. Compare this pattern to each existing theme


### DECISION PROCESS (MANDATORY)

Step 1: Evaluate whether ANY existing theme provides a strong conceptual match.

DEFAULT ASSUMPTION:
Assume that the code does NOT fit any existing theme unless strong evidence suggests otherwise.

A theme should only be selected if:
   - The code reflects the same underlying meaning as the theme.
   - The alignment between code and theme is based on meaning (not wording).
   - The match would be clear and unambiguous to an experienced qualitative researcher.

Step 2:
- If NO theme meets these criteria → return "no_fit"
- If ONE theme clearly meets these criteria → assign it

CRITICAL:
Assigning a code to an incorrect theme is a more serious error than returning "no_fit".

Return "no_fit" when:
- the fit is partial or ambiguous
- the similarity is based only on topic or keywords
- the pattern is novel or not represented
- the connection requires interpretation beyond the data


### OUTPUT REQUIREMENTS

- Return exactly one theme_label
- The value must be:
  - one of the existing themes
  - OR "no_fit"

- Provide a brief theme_justification (1–2 sentences):
  - explain why the theme fits to the code
  - OR explain why no theme fits to the code


### RULES

- Be conservative in assigning themes
- When uncertain, prefer "no_fit"
- Do NOT force a match
- Do NOT include any text outside the structured output
"""
    return content

### Defining  output schema with Pydantic  


In [ ]:
from pydantic import create_model, model_validator
from typing import Literal, Optional


def make_theme_model(themes: list[str]):
    cleaned = [t.strip() for t in themes if t and t.strip()]
    unique_themes = list(dict.fromkeys(cleaned))

    class ThemeAssignment(create_model(
        "ThemeAssignmentBase",
        theme_label=(Literal[*unique_themes, "no_fit"], ...),
        theme_justification=(Optional[str], None)
    )):

        @model_validator(mode="after")
        def check_justification(self):
            if self.theme_label != "no_fit" and not self.theme_justification:
                raise ValueError("theme_justification is required when theme_label is not 'no_fit'")
            return self

    return ThemeAssignment



In [ ]:
import json
import time

def run_with_retry_assign(outlines_model, prompt_content, response_model, tokenizer, max_retries=3, retry_delay=1):

    prompt_inst = format_prompt(prompt_content, tokenizer)

    for attempt in range(max_retries):
        try:

            # Outlines returns the validated object directly if passed a schema
            result = outlines_model(prompt_inst, response_model)

            # If outlines_model returns a string, keep these lines;
            # if it returns the object directly, just return result.
            parsed = json.loads(result) if isinstance(result, str) else result
            return response_model(**parsed) if isinstance(parsed, dict) else parsed

        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
                continue
            raise ValueError(f"Failed after {max_retries} attempts: {e}")

In [ ]:
def add_theme_to_themes(theme_label, code_uid, theme_justification, candidate_themes, themes):

    # skip non-theme labels
    if theme_label in {"no_fit", "Singular observation"}:
        print(f"no_fit or sigular observation found")
        return

    # safe fallback for justification
    theme_justification = theme_justification or "No justification provided"

    match = candidate_themes.loc[
        candidate_themes["theme_label"] == theme_label,
        "theme_description"
    ]

    # skip if label not found (extra safety)
    if match.empty:
        return

    if theme_label not in themes:
        themes[theme_label] = {
            "code_uids": [],
            "theme_description": match.iloc[0],
            "assignments": {}
        }

    if code_uid not in themes[theme_label]["code_uids"]:
        themes[theme_label]["code_uids"].append(code_uid)

    themes[theme_label]["assignments"][code_uid] = {
        "theme_justification": theme_justification
    }

In [ ]:
#Function for normalizing theme labels
CONTROL_LABELS = {"no_fit", "Singular observation"}

def normalize_theme_format(label: str) -> str:
    if not label:
        return None

    label = label.strip()

    # 🔹 protect control labels (case-insensitive)
    if label.lower() == "no_fit":
        return "no_fit"
    if label.lower() == "singular observation":
        return "Singular observation"

    # collapse whitespace
    label = re.sub(r"\s+", " ", label)

    # remove trailing punctuation
    label = label.strip(" .,-")

    # capitalize first letter
    if label:
        label = label[0].upper() + label[1:]

    return label

### Run pipeline - call functions



function der assigner tema til code

In [ ]:
def process_code_uid(
    uid,
    df, #main Dataframe across inteviews to allow changes mid-loop
    df_initial_themes, #Thematic map
    themes,
    reassess_bucket,
    tokenizer,
    outlines_model,
    THRESHOLD=10,
    verbose=True
):
    try:
        #
        row = df.loc[df["code_uid"] == uid].iloc[0]

        if verbose:
            print(f"Processing row {uid}")

        #
        if not df_initial_themes.empty:
            themes_str = "\n".join([
                f"- {r['theme_label']}: {r['theme_description']}"
                for _, r in df_initial_themes.iterrows()
            ])
        else:
            themes_str = "None"

        # Build prompt
        content = get_theme_prompt(
            row["codes_final"],
            row["code_uid"],
            row["justifications_final"],
            row["interview_excerpt"],
            themes_str
        )

        # Build response model
        response_model = make_theme_model(
            df_initial_themes["theme_label"].tolist()
            if not df_initial_themes.empty else []
        )

        #
        validated = run_with_retry_assign(
            outlines_model,
            content,
            response_model,
            tokenizer
        )

        if verbose:
            print(f"pydantic output: {validated}")

        theme_label = normalize_theme_format(validated.theme_label)
        theme_justification = validated.theme_justification

        # Handle "no_fit"
        if theme_label == 'no_fit':
            theme_justification = None
            reassess_bucket[uid] = reassess_bucket.get(uid, 0) + 1

            print(f"Item {uid} added to reassess_bucket")

            if reassess_bucket[uid] >= THRESHOLD:
                theme_label = "Singular observation"
                del reassess_bucket[uid]

                if verbose:
                    print(f"Item {uid} moved after {THRESHOLD} attempts")

        # Update themes
        add_theme_to_themes(theme_label, uid, theme_justification, df_initial_themes, themes)

        return {
            "code_uid": uid,
            "theme_label": theme_label,
            "theme_justification": theme_justification,
            "theme_error": None
        }

    except Exception as e:
        if verbose:
            print(f"DEBUG: Row {uid} failed with error: {type(e).__name__} - {e}")

        return {
            "code_uid": uid,
            "theme_label": None,
            "theme_justification": None,
            "theme_error": str(e)
        }

In [ ]:
import time
import pandas as pd

unique_int_ids = data["int_id"].unique()
total_interviews = len(unique_int_ids)
batch_size = 10

print(f"Starting thematic coding refine phase... ({total_interviews} interviews)\n")

# df_initial_themes contaimms candidate themes, uid_pool is the list of availible code_uids for usage in batch-func

start_total = time.time()
results = []
reassess_bucket = {}  # Dictionary: { uid: no of tries }
quarantine_bucket = [] # List for items that have failed too many times
THRESHOLD = 20 # limit for code not fitting to excising theme
REASSESS_BUCKET_SIZE = 20
themes = {}

for i, int_id in enumerate(unique_int_ids, start=1):
    interview_start = time.time()

    print(f"\n--- Interview {i}/{total_interviews}: {int_id} ---")

    df_subset = data[data["int_id"] == int_id]

    print(f"Number of excerpts: {len(df_subset)}")

    for uid in df_subset["code_uid"]:
        result = process_code_uid(
            uid=uid,
            df=df_subset,
            df_initial_themes=df_initial_themes,
            themes=themes,
            reassess_bucket=reassess_bucket,
            tokenizer=tokenizer,
            outlines_model=outlines_model,
            THRESHOLD=THRESHOLD
        )

        if result["theme_label"] != 'no_fit':
          results.append(result)

        if len(reassess_bucket)>REASSESS_BUCKET_SIZE:
          print(f"Max bucket size reached. Started new search for themes")
          new_initial_themes = run_theme_extraction(data,uid_pool,outlines_model=outlines_model, tokenizer=tokenizer, existing_candidate_themes=df_initial_themes)
          if new_initial_themes is not None and not new_initial_themes.empty:
            print(f"New themes found: {new_initial_themes}")
            df_initial_themes = (pd.concat([df_initial_themes, pd.DataFrame(new_initial_themes)], ignore_index=True).drop_duplicates(subset=["theme_label"]))



            # New themes have now been added, so run it against the updated thematic map
            print(f"Initating reassesment of codes in bucket")
            for uid in list(reassess_bucket):
              result = process_code_uid(
                  uid=uid,
                  df=data,
                  df_initial_themes=df_initial_themes,
                  themes=themes,
                  reassess_bucket=reassess_bucket,
                  tokenizer=tokenizer,
                  outlines_model=outlines_model,
                  THRESHOLD=THRESHOLD
              )

              #remove from bucket if theme found
              if result["theme_label"] != 'no_fit':
                del reassess_bucket[uid]

              results.append(result)



    interview_time = time.time() - interview_start
    print(f"Finished interview {int_id} in {round(interview_time, 2)} sec")

total_time = time.time() - start_total

results_df = pd.DataFrame(results)
themes_df = pd.DataFrame(themes)
print(results_df.head(10))
print(f"\nTotal time: {round(total_time, 2)} sec ({round(total_time/60, 2)} min)")

In [ ]:
# re-orient Themes_df for correct view
themes_df = pd.DataFrame.from_dict(themes, orient="index").reset_index()
themes_df = themes_df.rename(columns={"index": "theme_label"})

In [ ]:
themes_df = themes_df[
    ["theme_label", "theme_description", "code_uids"]
]

In [ ]:
themes_df["n_codes"] = themes_df["code_uids"].apply(len)

In [ ]:
no_fit_df = results_df[results_df["theme_label"] == "no_fit"]

In [ ]:
#Save Theme-level results
from datetime import datetime
import shutil

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
FINAL_DIR = pipeline1_path


# SAVE THEMES (theme-centric)

themes_parquet_ts = FINAL_DIR / f"themes_{timestamp}.parquet"
themes_parquet_current = FINAL_DIR / "current_themes.parquet"

themes_csv_ts = FINAL_DIR / f"themes_{timestamp}.csv"
themes_csv_current = FINAL_DIR / "current_themes.csv"

themes_df.to_parquet(themes_parquet_ts, index=False)
shutil.copy(themes_parquet_ts, themes_parquet_current)

themes_df.to_csv(themes_csv_ts, sep=";", index=False, encoding="utf-8-sig")
shutil.copy(themes_csv_ts, themes_csv_current)

print(f"Saved {len(themes_df)} themes")

Saved 10 themes


In [ ]:
#Save code-centric DF

# --- PARQUET ---
results_parquet_ts = FINAL_DIR / f"code_results_{timestamp}.parquet"
results_parquet_current = FINAL_DIR / "current_code_results.parquet"

results_df.to_parquet(results_parquet_ts, index=False)
shutil.copy(results_parquet_ts, results_parquet_current)

# CSV
results_csv_ts = FINAL_DIR / f"code_results_{timestamp}.csv"
results_csv_current = FINAL_DIR / "current_code_results.csv"

results_df.to_csv(results_csv_ts, sep=";", index=False, encoding="utf-8-sig")
shutil.copy(results_csv_ts, results_csv_current)

# LOG
print(f"Saved {len(results_df)} code assignments")

In [ ]:
#Save the no_fit Df

# PARQUET
no_fit_parquet_ts = FINAL_DIR / f"no_fit_{timestamp}.parquet"
no_fit_parquet_current = FINAL_DIR / "current_no_fit.parquet"

no_fit_df.to_parquet(no_fit_parquet_ts, index=False)
shutil.copy(no_fit_parquet_ts, no_fit_parquet_current)

# CSV
no_fit_csv_ts = FINAL_DIR / f"no_fit_{timestamp}.csv"
no_fit_csv_current = FINAL_DIR / "current_no_fit.csv"

no_fit_df.to_csv(no_fit_csv_ts, sep=";", index=False, encoding="utf-8-sig")
shutil.copy(no_fit_csv_ts, no_fit_csv_current)

print(f"Saved {len(no_fit_df)} codes with no fit")